In [1]:
import paramiko
import re
import os
import time
from IPython.display import clear_output

# Paths & Cluster Config

Edit the paths below to match your cluster setup before running.

In [ ]:
# Cluster login
CLUSTER_HOST = 'loginserver.elsc.huji.ac.il'
USERNAME = 'qixin.yang'
SSH_KEY_PATH = os.path.expanduser('~/.ssh/id_rsa')  # set to None to use SSH agent

# Repo root on the cluster
PIPELINE_WORKDIR = '/ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline'
sh_script = f'{PIPELINE_WORKDIR}/utils/run_python_job.sh'
CONDA_ENV = 'adamlab_pipeline'  # Python 3.11

# Analysis paths on the cluster (inside the repo)
ANALYSIS_ROOT = f'{PIPELINE_WORKDIR}/miniVI_PlaceCell_analysis_V4'
DATA_ROOT     = f'{ANALYSIS_ROOT}/data'
FIGURES_ROOT  = f'{ANALYSIS_ROOT}/figures'
LOG_DIR       = f'{ANALYSIS_ROOT}/logs'

# ===== Analysis parameters (change these between runs) =====
DIRECTION_MODE  = 'head'         # 'head' or 'travel'
SPIKE_TYPE      = 'all_spike'    # 'all_spike', 'simple_spike', or 'complex_spike'
N_SURROGATES    = 1000
FIRST_N_MINUTES = 10.0
FORCE_RECOMPUTE = False

# Shared manifest directory (same for all direction/spike combos)
# Only need to run Step 1 once — the manifest is reused across runs.
MANIFEST_DIR = f'{FIGURES_ROOT}/CKII_pooled/egocentric_tuning_carpenter'

# Output directory: encodes direction + spike type so results don't overwrite
# e.g. .../egocentric_tuning_carpenter/head_all_spike/
OUTPUT_DIR = f'{MANIFEST_DIR}/{DIRECTION_MODE}_{SPIKE_TYPE}'

# Per-cell job resources
CELL_CPUS   = 10       # CPUs per cell task (for surrogate parallelism)
CELL_MEM_GB = 32      # memory per cell task

# Script paths (relative to PIPELINE_WORKDIR)
PREPARE_SCRIPT   = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_prepare.py'
CELL_SCRIPT      = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_single_cell.py'
AGGREGATE_SCRIPT = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/aggregate_egocentric_results.py'
PLOT_SCRIPT      = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_plot_cells.py'
STATS_SCRIPT     = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_summary_stats.py'

print(f'Run config:    {DIRECTION_MODE} direction, {SPIKE_TYPE} spikes')
print(f'Conda env:     {CONDA_ENV}')
print(f'Manifest dir:  {MANIFEST_DIR}')
print(f'Output dir:    {OUTPUT_DIR}')

# Connect to Cluster

In [9]:
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
kwargs = {}
if SSH_KEY_PATH and os.path.exists(SSH_KEY_PATH):
    kwargs['key_filename'] = SSH_KEY_PATH
ssh.connect(CLUSTER_HOST, username=USERNAME, **kwargs)
print('[INFO] Connected to cluster.')

def to_local(cluster_path):
    """Convert cluster path to local Mac path via mounted network volume."""
    return cluster_path.replace('/ems/elsc-labs/adam-y', '/Volumes/adam-lab')

def run_command(command):
    """Run a command on the cluster with a login shell."""
    stdin, stdout, stderr = ssh.exec_command(f"bash -l -c '{command}'")
    output = stdout.read().decode().strip()
    error  = stderr.read().decode().strip()
    return output, error

def wait_for_jobs(job_ids, poll_interval=60):
    """Poll SLURM until all jobs complete. Returns True if all succeeded."""
    if not job_ids:
        print('No jobs to wait for.')
        return True
    pending_jobs = set(job_ids)
    failed_jobs = []
    while pending_jobs:
        job_list = ','.join(pending_jobs)
        output, error = run_command(f'sacct -j {job_list} --format=JobID,State,ExitCode -n -P')
        if error and 'Invalid job id' not in error:
            print(f'[WARNING] {error}')
        completed = set()
        for line in output.strip().splitlines():
            if not line or '.' in line.split('|')[0]:
                continue
            parts = line.split('|')
            if len(parts) >= 2:
                job_id, state = parts[0], parts[1]
                if state in ['COMPLETED', 'FAILED', 'CANCELLED', 'TIMEOUT']:
                    completed.add(job_id)
                    if state != 'COMPLETED':
                        failed_jobs.append((job_id, state))
        pending_jobs -= completed
        clear_output(wait=True)
        print(f'[{time.strftime("%H:%M:%S")}] Jobs status:')
        print(f'  Completed: {len(job_ids) - len(pending_jobs)}/{len(job_ids)}')
        print(f'  Pending:   {len(pending_jobs)}')
        if failed_jobs:
            print(f'  Failed:    {failed_jobs}')
        if pending_jobs:
            print(f'\nWaiting {poll_interval}s before next check...')
            time.sleep(poll_interval)
    print('\n' + '=' * 50)
    if failed_jobs:
        print(f'WARNING: {len(failed_jobs)} job(s) failed: {failed_jobs}')
        return False
    print('All jobs completed successfully!')
    return True

[INFO] Connected to cluster.


# Step 0: Setup Data Links (run once)

Creates symlinks from `data/ANIMAL_NAME/merged_aligned_data.pkl` to the actual files on the HPC network.
Safe to re-run — existing links are skipped.

In [ ]:
setup_script = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/setup_data_links.py'

_, mkdir_err = run_command(f'mkdir -p {LOG_DIR}')
if mkdir_err:
    print(f'[WARN] mkdir: {mkdir_err}')

cmd = (
    f'sbatch --job-name ego_setup -c 1 --mem=4G -t 00:05:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {setup_script}'
)

output, error = run_command(cmd)
setup_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        setup_job_id = match.group(1)
        print(f'[INFO] Setup Job ID: {setup_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_setup_{setup_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_setup_{setup_job_id}.err')

if setup_job_id:
    wait_for_jobs([setup_job_id], poll_interval=15)
else:
    print('[INFO] No setup job submitted.')

# Step 1: Prepare — Build Cache & Cell Manifest (run once)

Ensures per-animal cache exists, classifies cells, and writes a `manifest.json`
listing all cells to analyze. The manifest is saved to the **shared** `MANIFEST_DIR`
(not the direction/spike-specific output dir), so you only need to run this once —
then re-run Steps 2–3 with different `DIRECTION_MODE` / `SPIKE_TYPE`.

In [4]:
_, mkdir_err = run_command(f'mkdir -p {LOG_DIR} {MANIFEST_DIR}')
if mkdir_err:
    print(f'[WARN] mkdir: {mkdir_err}')

force_flag = '--force-recompute' if FORCE_RECOMPUTE else ''

cmd = (
    f'sbatch --job-name ego_prepare -c 4 --mem=32G -t 02:00:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {PREPARE_SCRIPT} '
    f'--data-root {DATA_ROOT} '
    f'--figures-root {FIGURES_ROOT} '
    f'--output-dir {MANIFEST_DIR} '
    f'--categories CSplus CSminus all-nonPLC '
    f'{force_flag}'
)

output, error = run_command(cmd)
prepare_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        prepare_job_id = match.group(1)
        print(f'[INFO] Prepare Job ID: {prepare_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_prepare_{prepare_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_prepare_{prepare_job_id}.err')

[INFO] Submitted batch job 28151305
[INFO] Prepare Job ID: 28151305
[INFO] Log .out: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_prepare_28151305.out
[INFO] Log .err: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/ego_prepare_28151305.err


In [5]:
# Wait for prepare job to finish
if prepare_job_id:
    wait_for_jobs([prepare_job_id], poll_interval=30)
else:
    print('[INFO] No prepare job submitted.')

[17:41:30] Jobs status:
  Completed: 1/1
  Pending:   0

All jobs completed successfully!


# Step 2: Submit Array Job — One Task Per Cell

Each SLURM array task processes one cell independently.
All tasks run in parallel across the cluster.

In [10]:
# Read manifest to get cell count
import json

manifest_output, manifest_error = run_command(f'cat {MANIFEST_DIR}/manifest.json')
if manifest_error:
    print(f'[ERROR] {manifest_error}')
    N_CELLS = 0
else:
    manifest = json.loads(manifest_output)
    N_CELLS = len(manifest)
    print(f'Total cells to analyze: {N_CELLS}')
    for cat in set(m['category'] for m in manifest):
        n = sum(1 for m in manifest if m['category'] == cat)
        print(f'  {cat}: {n}')

Total cells to analyze: 56
  CSplus: 12
  all-nonPLC: 36
  CSminus: 8


In [11]:
if N_CELLS == 0:
    print('[ERROR] No cells in manifest. Check Step 1.')
    array_job_id = None
else:
    cmd = (
        f'sbatch --job-name ego_cell '
        f'--array=0-{N_CELLS - 1} '
        f'-c {CELL_CPUS} --mem={CELL_MEM_GB}G '
        f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
        f'--chdir {PIPELINE_WORKDIR} '
        f'--output {LOG_DIR}/ego_cell_%A_%a.out '
        f'--error {LOG_DIR}/ego_cell_%A_%a.err '
        f'{sh_script} {PIPELINE_WORKDIR} {CELL_SCRIPT} '
        f'--output-dir {OUTPUT_DIR} '
        f'--manifest-dir {MANIFEST_DIR} '
        f'--data-root {DATA_ROOT} '
        f'--figures-root {FIGURES_ROOT} '
        f'--direction-mode {DIRECTION_MODE} '
        f'--spike-type {SPIKE_TYPE} '
        f'--n-surrogates {N_SURROGATES} '
        f'--n-jobs {CELL_CPUS} '
        f'--first-n-minutes {FIRST_N_MINUTES}'
    )

    output, error = run_command(cmd)
    array_job_id = None
    if error:
        print(f'[ERROR] {error}')
    else:
        print(f'[INFO] {output}')
        match = re.search(r'Submitted batch job (\d+)', output)
        if match:
            array_job_id = match.group(1)
            print(f'[INFO] Array Job ID: {array_job_id}')
            print(f'[INFO] {N_CELLS} tasks submitted (0 to {N_CELLS - 1})')
            print(f'[INFO] Each task: {CELL_CPUS} CPUs, {CELL_MEM_GB}GB memory')
            print(f'[INFO] Config: {DIRECTION_MODE} direction, {SPIKE_TYPE} spikes')
            print(f'[INFO] Log dir (local): {to_local(LOG_DIR)}/')
            print(f'[INFO] Log pattern: ego_cell_{array_job_id}_<task_id>.out/.err')

[INFO] Submitted batch job 28151573
[INFO] Array Job ID: 28151573
[INFO] 56 tasks submitted (0 to 55)
[INFO] Each task: 10 CPUs, 32GB memory
[INFO] Config: head direction, all_spike spikes
[INFO] Log dir (local): /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/logs/
[INFO] Log pattern: ego_cell_28151573_<task_id>.out/.err


In [7]:
# Wait for all array tasks to complete
if array_job_id:
    # For array jobs, monitor the main job ID
    wait_for_jobs([array_job_id], poll_interval=30)
else:
    print('[INFO] No array job submitted.')

[17:57:12] Jobs status:
  Completed: 0/1
  Pending:   1

Waiting 30s before next check...


KeyboardInterrupt: 

# Step 3: Aggregate Results

Collects all per-cell `.npz` results into:
- `egocentric_tuning_summary.csv`
- `egocentric_tuning_skipped.csv`
- `null_distributions/` (optional)

In [ ]:
dep = f'--dependency=afterany:{array_job_id} ' if array_job_id else ''

cmd = (
    f'sbatch {dep}--job-name ego_aggregate -c 1 --mem=8G -t 00:10:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {AGGREGATE_SCRIPT} '
    f'--output-dir {OUTPUT_DIR} '
    f'--save-null-distributions'
)

output, error = run_command(cmd)
agg_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        agg_job_id = match.group(1)
        print(f'[INFO] Aggregate Job ID: {agg_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_aggregate_{agg_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_aggregate_{agg_job_id}.err')
        print(f'[INFO] Results dir (local): {to_local(OUTPUT_DIR)}/')

In [ ]:
if agg_job_id:
    wait_for_jobs([agg_job_id], poll_interval=30)
else:
    print('[INFO] No aggregate job submitted.')

# Step 4: Check Results

In [ ]:
# Check the aggregate output
if agg_job_id:
    log_path = f'{LOG_DIR}/ego_aggregate_{agg_job_id}.out'
    output, error = run_command(f'tail -30 {log_path}')
    print(output)
    if error:
        print(f'[ERROR] {error}')
else:
    print('[INFO] No aggregate job ID.')

In [ ]:
# Preview the summary CSV
output, error = run_command(f'head -10 {OUTPUT_DIR}/egocentric_tuning_summary.csv')
if error:
    print(f'[ERROR] {error}')
else:
    print('Summary CSV (first 10 rows):')
    print(output)

print()
output, error = run_command(f'wc -l {OUTPUT_DIR}/egocentric_tuning_summary.csv {OUTPUT_DIR}/egocentric_tuning_skipped.csv')
print(output)

# Step 5: Plot Results

## 5a — Per-cell summary plots
Generates a multi-panel summary figure for each cell (trajectory + spikes colored by direction,
spatial maps with fitted arrows, preferred/non-preferred split maps for all-spikes/SS/CS/theta/slow-Vm).
Figures are organized by category folder: `per_cell_summary/{CSplus,CSminus,all-nonPLC}/`.

## 5b — Summary statistics
Compares pass counts and mean resultant length (MRL) across **all three categories** (CSplus, CSminus, all-nonPLC)
with pairwise Mann-Whitney U tests. Saves bar charts, box plots, and CSV tables.

In [ ]:
# Step 5a: Per-cell summary plots
# Runs as a single job (caches data per animal, so much faster than array jobs).
# Needs enough memory to hold one animal's merged data at a time.

cmd = (
    f'sbatch --job-name ego_plot -c 4 --mem=64G -t 04:00:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {PLOT_SCRIPT} '
    f'--output-dir {OUTPUT_DIR} '
    f'--manifest-dir {MANIFEST_DIR} '
    f'--data-root {DATA_ROOT} '
    f'--figures-root {FIGURES_ROOT} '
    f'--direction-mode {DIRECTION_MODE} '
    f'--first-n-minutes {FIRST_N_MINUTES} '
    f'--save-formats svg png'
)

output, error = run_command(cmd)
plot_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        plot_job_id = match.group(1)
        print(f'[INFO] Plot Job ID: {plot_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_plot_{plot_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_plot_{plot_job_id}.err')
        print(f'[INFO] Figures dir (local): {to_local(OUTPUT_DIR)}/per_cell_summary/')

In [ ]:
# Wait for per-cell plots to finish
if plot_job_id:
    wait_for_jobs([plot_job_id], poll_interval=60)
else:
    print('[INFO] No plot job submitted.')

In [ ]:
# Step 5b: Summary statistics (CSplus vs CSminus vs all-nonPLC)
# Lightweight job — just reads the summary CSV and makes plots.

dep = f'--dependency=afterany:{plot_job_id} ' if plot_job_id else ''

cmd = (
    f'sbatch {dep}--job-name ego_stats -c 1 --mem=4G -t 00:10:00 '
    f'--export=ALL,CONDA_ENV_NAME={CONDA_ENV} '
    f'--chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {STATS_SCRIPT} '
    f'--output-dir {OUTPUT_DIR} '
    f'--categories CSplus CSminus all-nonPLC '
    f'--save-formats svg png'
)

output, error = run_command(cmd)
stats_job_id = None
if error:
    print(f'[ERROR] {error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        stats_job_id = match.group(1)
        print(f'[INFO] Stats Job ID: {stats_job_id}')
        print(f'[INFO] Log .out: {to_local(LOG_DIR)}/ego_stats_{stats_job_id}.out')
        print(f'[INFO] Log .err: {to_local(LOG_DIR)}/ego_stats_{stats_job_id}.err')
        print(f'[INFO] Stats dir (local): {to_local(OUTPUT_DIR)}/summary_stats/')

In [ ]:
# Wait for stats job and check results
if stats_job_id:
    wait_for_jobs([stats_job_id], poll_interval=15)

    # Print stats log
    log_path = f'{LOG_DIR}/ego_stats_{stats_job_id}.out'
    output, error = run_command(f'cat {log_path}')
    if output:
        print(output)
    if error:
        print(f'[ERROR] {error}')
else:
    print('[INFO] No stats job submitted.')

In [ ]:
# Check per-cell plot outputs
output, error = run_command(
    f'echo "=== Per-cell plot folders ===" && '
    f'ls -la {OUTPUT_DIR}/per_cell_summary/ 2>/dev/null && '
    f'echo "" && '
    f'for d in {OUTPUT_DIR}/per_cell_summary/*/; do '
    f'  name=$(basename "$d"); '
    f'  count=$(ls "$d"/*.svg 2>/dev/null | wc -l); '
    f'  echo "  $name: $count SVG files"; '
    f'done && '
    f'echo "" && '
    f'echo "=== Summary stats ===" && '
    f'ls -la {OUTPUT_DIR}/summary_stats/ 2>/dev/null'
)
print(output)
if error:
    print(f'[WARN] {error}')